# 01 — EDA + stratified split

Goals of this notebook:
1. Look at the raw Kaggle job-postings CSV.
2. Drop the 10 pre-engineered numeric columns (we cannot reproduce their formula, so we will recompute our own raw-text features in `src/features.py` at STEP 4).
3. Plot class balance and description-length distributions.
4. Sample 30 fake + 30 real postings to disk for qualitative review.
5. Clean: drop empty descriptions, drop duplicate descriptions, lowercase email domains.
6. Build the tagged `job_text` field that the embedder and the LLM will see.
7. Stratified 80 / 10 / 10 split, saved to `data/processed/`.

The raw CSV is gitignored; the processed splits are tracked so the train/val/test boundary is reproducible.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw' / 'aligned_kaggle_data.csv'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

# fixed seed so the same posting always lands in the same split
SEED = 42

## Load

In [ ]:
df = pd.read_csv(RAW)
# fraudulent is 0/1 in the source; cast explicitly so downstream comparisons cannot break
df['fraudulent'] = df['fraudulent'].astype(int)
print(f'rows: {len(df)}, columns: {len(df.columns)}')
df.head(2)

## Drop the pre-engineered numeric columns

These were already z-score-standardised in the source CSV. We cannot reproduce or defend their formula in viva, and STEP 4 (`src/features.py`) recomputes our own heuristic features from raw text where every line is explainable.

In [ ]:
PRE_ENGINEERED = [
    'desc_length', 'req_length', 'profile_length',
    'kw_work_from_home', 'kw_quick_money', 'kw_no_experience',
    'kw_urgent', 'kw_immediate_start',
    'desc_missing', 'profile_missing',
]
df = df.drop(columns=PRE_ENGINEERED)
print(f'after drop: {df.shape}')
df.dtypes

## Class balance

We expect the classic ~95 / 5 split for this dataset — important to confirm before we plan any rebalancing later.

In [ ]:
counts = df['fraudulent'].value_counts().sort_index()
print(counts)
print(f'fraud rate: {counts[1] / len(df):.2%}')

counts.plot(kind='bar')
plt.title('Fraudulent vs real postings')
plt.xlabel('fraudulent (0 = real, 1 = fraud)')
plt.ylabel('count')
plt.show()

## Description length distribution

If real and fraud postings have very different length distributions, length alone could leak the label — worth knowing before we build features.

In [ ]:
desc_len = df['description'].fillna('').str.len()
real_len = desc_len[df['fraudulent'] == 0]
fraud_len = desc_len[df['fraudulent'] == 1]

print(f'real    median chars: {real_len.median():.0f}   mean: {real_len.mean():.0f}')
print(f'fraud   median chars: {fraud_len.median():.0f}   mean: {fraud_len.mean():.0f}')

# clip the long tail so the bulk of the distribution is visible
plt.hist(real_len.clip(upper=5000), bins=40, alpha=0.6, label='real')
plt.hist(fraud_len.clip(upper=5000), bins=40, alpha=0.6, label='fraud')
plt.xlabel('description length (chars, clipped at 5000)')
plt.ylabel('count')
plt.legend()
plt.title('Description length by class')
plt.show()

## Cleaning

Three steps:
1. Drop rows where `description` is empty — we cannot score a posting with no text.
2. Drop duplicate descriptions — some scams are reposted verbatim under different `job_id`s; keeping copies would leak the same text across train / val / test.
3. Lowercase the domain part of any email — so `Recruiter@Gmail.COM` and `recruiter@gmail.com` look the same to the embedder and the keyword features.

In [ ]:
before = len(df)
df = df[df['description'].fillna('').str.strip() != '']
print(f'dropped {before - len(df)} rows with empty description')

before = len(df)
df = df.drop_duplicates(subset=['description'], keep='first')
print(f'dropped {before - len(df)} duplicate-description rows')

EMAIL_RE = re.compile(r'(\b[\w.+-]+@)([\w.-]+\.\w+)\b')

def lowercase_email_domains(text):
    if not isinstance(text, str):
        return text
    return EMAIL_RE.sub(lambda m: m.group(1) + m.group(2).lower(), text)

TEXT_COLS = ['title', 'company_profile', 'description', 'requirements', 'benefits']
for col in TEXT_COLS:
    df[col] = df[col].apply(lowercase_email_domains)

print(f'after cleaning: {df.shape}')

## Qualitative sample (30 fake + 30 real)

Saved to `data/processed/eda_samples.csv` so we have a fixed pool of postings to read by hand for the qualitative-analysis section of the report.

In [ ]:
fake_sample = df[df['fraudulent'] == 1].sample(n=30, random_state=SEED)
real_sample = df[df['fraudulent'] == 0].sample(n=30, random_state=SEED)
samples = pd.concat([fake_sample, real_sample], ignore_index=True)

samples.to_csv(PROCESSED / 'eda_samples.csv', index=False)
print(f"saved 30 fake + 30 real to {PROCESSED / 'eda_samples.csv'}")
samples[['job_id', 'title', 'fraudulent']].head(10)

## Build the tagged `job_text` field

This is the single string the embedder and the LLM will see for each posting. We label each section so the model knows which part is the title, which is the description, etc. Empty fields are skipped — no point training the model to ignore `[BENEFITS] ` noise.

In [ ]:
FIELD_TAGS = [
    ('title', 'TITLE'),
    ('location', 'LOCATION'),
    ('employment_type', 'EMPLOYMENT TYPE'),
    ('industry', 'INDUSTRY'),
    ('company_profile', 'COMPANY PROFILE'),
    ('description', 'DESCRIPTION'),
    ('requirements', 'REQUIREMENTS'),
    ('benefits', 'BENEFITS'),
    ('salary_range', 'SALARY'),
]

def build_job_text(row):
    parts = []
    for col, tag in FIELD_TAGS:
        v = row.get(col)
        if isinstance(v, str) and v.strip():
            parts.append(f'[{tag}] {v.strip()}')
    return '\n'.join(parts)

df['job_text'] = df.apply(build_job_text, axis=1)

print('example job_text (first fraud posting):')
print(df.loc[df['fraudulent'] == 1, 'job_text'].iloc[0][:600])

## Stratified 80 / 10 / 10 split

We split twice: first carve off 80 % for train, then split the remaining 20 % in half for val and test. Stratifying on `fraudulent` keeps the same ~5 % fraud rate in every split, so the val / test F1 we report later is honest with respect to real-world prevalence.

Train rebalancing (oversampling fraud toward 50 / 50) is **not** done here — it happens at STEP 8 when we pick the 1500 examples to send to the teacher model.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df['fraudulent'], random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['fraudulent'], random_state=SEED,
)

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    rate = part['fraudulent'].mean()
    print(f'{name:5s}  rows: {len(part):>5}   fraud rate: {rate:.2%}')

## Save splits

In [ ]:
train_df.to_csv(PROCESSED / 'train.csv', index=False)
val_df.to_csv(PROCESSED / 'val.csv', index=False)
test_df.to_csv(PROCESSED / 'test.csv', index=False)
print('wrote train.csv, val.csv, test.csv to data/processed/')